# Revisao: Redes Neurais com MLPClassifier
Notebook curto para treinar, salvar e usar uma rede neural simples.

In [ ]:
# Pandas para montar tabelas (DataFrame)
import pandas as pd

# Joblib para salvar e carregar o modelo treinado
import joblib

# Path para criar pasta/arquivo de forma organizada
from pathlib import Path

# MLPClassifier: rede neural de multiplas camadas do scikit-learn
from sklearn.neural_network import MLPClassifier

# Ferramentas de preparacao dos dados
from sklearn.preprocessing import MinMaxScaler

## Dados de Treino
Mesma base simples do Perceptron: renda e historico de pagamento decidem aprovacao de credito.

In [ ]:
# Dados de treino: cada linha e um cliente com renda, historico e resultado real
df_train = pd.DataFrame([
    {"renda": 8.0, "historico": 9.0, "aprovado": 1},
    {"renda": 3.0, "historico": 2.0, "aprovado": 0},
    {"renda": 6.0, "historico": 7.0, "aprovado": 1},
    {"renda": 2.0, "historico": 5.0, "aprovado": 0},
    {"renda": 9.0, "historico": 8.0, "aprovado": 1},
    {"renda": 1.0, "historico": 1.0, "aprovado": 0},
    {"renda": 7.0, "historico": 6.0, "aprovado": 1},
    {"renda": 4.0, "historico": 3.0, "aprovado": 0},
])

# Separa entradas (X) da saida esperada (y)
X_train = df_train[["renda", "historico"]]
y_train = df_train["aprovado"]

print(df_train)

## Preparacao e Treino
Antes de treinar, normalizamos os dados com MinMaxScaler (mesmo processo da aula anterior).
Depois, treinamos o MLPClassifier com uma camada oculta de 5 neuronios.

In [ ]:
# Normaliza os dados entre 0 e 1 para o modelo aprender melhor
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_train)

# Cria a rede neural com uma camada oculta de 5 neuronios
# hidden_layer_sizes=(5,) significa: 1 camada oculta, 5 neuronios dentro dela
# max_iter: numero maximo de ajustes internos; aumentar evita o ConvergenceWarning
modelo = MLPClassifier(hidden_layer_sizes=(5,), max_iter=2000, random_state=42)

# fit = a rede aprende os padroes dos dados de treino
modelo.fit(X_scaled, y_train)

print("Modelo treinado com sucesso.")

## Salvando o Modelo
Igual ao pipeline da aula anterior: salvamos tudo para reutilizar sem precisar treinar de novo.

In [ ]:
# Cria a pasta models/ se nao existir
Path("models").mkdir(exist_ok=True)

# Salva o modelo e o scaler juntos para garantir consistencia na previsao
joblib.dump({"modelo": modelo, "scaler": scaler}, "models/revisao_mlp.joblib")

print("Modelo salvo em models/revisao_mlp.joblib")

## Usando o Modelo em Novos Dados
Aqui simulamos o que acontece em producao: carregamos o modelo salvo e fazemos a previsao.
Nao ha fit de novo. So carregamos e usamos predict().

In [ ]:
# Novos clientes chegando para avaliacao
df_novo = pd.DataFrame([
    {"renda": 7.5, "historico": 8.0},
    {"renda": 2.0, "historico": 3.5},
])

# Carrega o modelo e o scaler salvos
state = joblib.load("models/revisao_mlp.joblib")
modelo_prod = state["modelo"]
scaler_prod = state["scaler"]

# Aplica a mesma normalizacao do treino (sem reaprender)
X_novo = scaler_prod.transform(df_novo)

# predict = aplica o que foi aprendido no fit, sem retreinar
previsoes = modelo_prod.predict(X_novo)

df_novo["aprovado_previsto"] = previsoes
df_novo

## Como isso se aproxima da realidade?
O `aprovado_previsto` e o resultado que a rede neural aprendeu a gerar a partir dos padroes do treino.

Em producao, esse valor seria usado para tomar uma decisao no sistema: aprovar ou reprovar o cliente automaticamente, sem nenhum humano no meio.

O fluxo real e sempre o mesmo:
- `fit()` acontece uma vez, no treino
- o modelo e salvo
- `predict()` e usado para cada novo dado que chega

---

## Parte 2: Organizando com uma Classe

Ate aqui fizemos tudo solto. Na pratica, o codigo fica dentro de uma classe para ser reutilizado facilmente em qualquer projeto.

A logica e a mesma do `MiniFeatureEngineer` da aula anterior: treinar, salvar, carregar e prever.

In [ ]:
class MiniMLPClassifier:
    # Classe pequena para ensinar treino, salvamento, carregamento e previsao
    def __init__(self, model_dir="models", nome="mini_mlp"):
        self.model_dir = Path(model_dir)
        self.model_dir.mkdir(parents=True, exist_ok=True)  # cria a pasta se nao existir
        self.path = self.model_dir / f"{nome}.joblib"

        # Comeca vazio: sera preenchido no treinar() ou carregar()
        self.modelo = None
        self.scaler = None

    def treinar(self, X, y, neuronios=(10,)):
        # Normaliza os dados entre 0 e 1 antes de treinar
        self.scaler = MinMaxScaler()
        X_scaled = self.scaler.fit_transform(X)

        # Cria e treina a rede neural com a arquitetura informada
        self.modelo = MLPClassifier(hidden_layer_sizes=neuronios, max_iter=2000, random_state=42)
        self.modelo.fit(X_scaled, y)

    def prever(self, X):
        # Se nao houver modelo em memoria, carrega do arquivo
        if self.modelo is None:
            self.carregar()

        # Aplica a mesma normalizacao do treino, sem reaprender
        X_scaled = self.scaler.transform(X)
        return self.modelo.predict(X_scaled)

    def salvar(self):
        # Salva tudo que foi aprendido no treino
        joblib.dump({"modelo": self.modelo, "scaler": self.scaler}, self.path)

    def carregar(self):
        # Recarrega o estado salvo para usar em producao
        state = joblib.load(self.path)
        self.modelo = state["modelo"]
        self.scaler = state["scaler"]

In [ ]:
# Usando a classe com os mesmos dados de credito da Parte 1
clf = MiniMLPClassifier(nome="credito")
clf.treinar(X_train, y_train)
clf.salvar()

print("Modelo treinado e salvo pela classe.")

# Novos clientes
df_novos = pd.DataFrame([
    {"renda": 9.0, "historico": 8.5},
    {"renda": 1.5, "historico": 2.0},
])

resultado = clf.prever(df_novos)
df_novos["aprovado"] = resultado
df_novos

---

## Parte 3: Exemplo Tabular Real — Classificação de Vinhos

Agora usamos um dataset real que ja vem dentro do scikit-learn: o `load_wine`.

Ele tem dados quimicos de 178 vinhos e o objetivo e classificar qual dos 3 tipos de uva originou cada vinho.
Ninguem precisa entender quimica: a rede neural aprende os padroes sozinha.

Isso e exatamente o que acontece em industrias onde sensores medem propriedades quimicas e um modelo classifica o produto automaticamente.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

# Carrega o dataset de vinhos (ja vem dentro do scikit-learn, sem download)
wine = load_wine()

# Converte para DataFrame para visualizar
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine["tipo"] = wine.target  # 0, 1 ou 2 representam os 3 tipos de uva

print(f"Total de vinhos: {len(df_wine)}")
print(f"Tipos de vinho: {wine.target_names}")
df_wine.head()

In [ ]:
# Separa treino e teste: 80% para treinar, 20% para avaliar
X_wine = df_wine.drop(columns="tipo")
y_wine = df_wine["tipo"]

X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42
)

# Treina usando a classe que acabamos de criar
clf_wine = MiniMLPClassifier(nome="vinho")
clf_wine.treinar(X_wine_train, y_wine_train, neuronios=(20, 10))  # 2 camadas ocultas
clf_wine.salvar()

# Avalia no conjunto de teste
from sklearn.metrics import accuracy_score

previsoes_wine = clf_wine.prever(X_wine_test)
acuracia = accuracy_score(y_wine_test, previsoes_wine)
print(f"Acuracia no conjunto de teste: {acuracia:.0%}")

In [ ]:
# Compara as previsoes com os valores reais de forma legivel
df_resultado = X_wine_test.copy()
df_resultado["tipo_real"] = [wine.target_names[i] for i in y_wine_test]
df_resultado["tipo_previsto"] = [wine.target_names[i] for i in previsoes_wine]
df_resultado["correto"] = df_resultado["tipo_real"] == df_resultado["tipo_previsto"]

# Mostra apenas as colunas importantes para a comparacao
df_resultado[["tipo_real", "tipo_previsto", "correto"]].head(10)

---

## Parte 4: Reconhecendo Dígitos com Imagens

Este e o exemplo mais proximo do que a IA faz no mundo real.

O scikit-learn tem um dataset de digitos manuscritos chamado `load_digits`.
Cada imagem e uma grade de 8x8 pixels (64 numeros) representando um digito de 0 a 9.

A rede neural vai aprender a identificar qual digito e qual olhando apenas para esses 64 numeros, sem nenhuma regra escrita por nos.

Aplicacoes reais: leitura de CEP em envelopes, reconhecimento de placas, leitura de cheques bancarios.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# Carrega o dataset de digitos (ja vem dentro do scikit-learn, sem download)
digits = load_digits()

print(f"Total de imagens: {len(digits.images)}")
print(f"Tamanho de cada imagem: {digits.images[0].shape} pixels")
print(f"Digitos possiveis: {digits.target_names}")

# Mostra as primeiras 10 imagens para ver como sao os dados
fig, eixos = plt.subplots(2, 5, figsize=(10, 4))
for i, eixo in enumerate(eixos.flat):
    eixo.imshow(digits.images[i], cmap="gray")
    eixo.set_title(f"Digito: {digits.target[i]}")
    eixo.axis("off")

plt.suptitle("Primeiras 10 imagens do dataset", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cada imagem de 8x8 vira uma linha de 64 numeros (o modelo nao ve imagem, ve numeros)
X_digits = digits.images.reshape(len(digits.images), -1)
y_digits = digits.target

print(f"Formato de entrada para o modelo: {X_digits.shape}")
print(f"Exemplo de uma imagem como numeros:\n{X_digits[0]}")  # 64 valores de 0 a 16

# Separa treino e teste
X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42
)

In [ ]:
# Treina a rede neural para reconhecer os digitos
# Usamos 2 camadas ocultas com 64 neuronios cada (mesma quantidade de pixels da entrada)
clf_digits = MiniMLPClassifier(nome="digitos")
clf_digits.treinar(X_d_train, y_d_train, neuronios=(64, 64))
clf_digits.salvar()

# Avalia no conjunto de teste
previsoes_digits = clf_digits.prever(X_d_test)
acuracia_digits = accuracy_score(y_d_test, previsoes_digits)
print(f"Acuracia no reconhecimento de digitos: {acuracia_digits:.0%}")

In [ ]:
# Visualiza 10 imagens do conjunto de teste com a previsao da rede neural
fig, eixos = plt.subplots(2, 5, figsize=(10, 4))
for i, eixo in enumerate(eixos.flat):
    # Recupera a imagem original em formato 8x8
    imagem = X_d_test[i].reshape(8, 8)
    real = y_d_test[i]
    previsto = previsoes_digits[i]
    correto = "OK" if real == previsto else "ERRO"

    eixo.imshow(imagem, cmap="gray")
    eixo.set_title(f"Real:{real}  Prev:{previsto}  {correto}", fontsize=8)
    eixo.axis("off")

plt.suptitle("Rede neural reconhecendo digitos manuscritos", fontsize=13)
plt.tight_layout()
plt.show()

## O que acabamos de fazer?

Treinamos uma rede neural que le imagens de digitos e identifica qual numero e.

O modelo nunca recebeu uma regra dizendo "o 7 tem essa forma". Ele aprendeu sozinho olhando para os padroes dos 64 pixels em cada imagem.

Isso e exatamente o principio por tras de sistemas como:
- leitura automatica de CEP nos Correios
- reconhecimento de placa em pedagios
- leitura de cheques e documentos escaneados
- captcha e validacao de formularios

O fluxo e sempre o mesmo: `treinar()` uma vez, `salvar()`, e depois `prever()` para cada nova imagem que chega.

---

## Bonus: Testando com uma Imagem Real da Internet

A rede foi treinada com imagens de 8x8 pixels. Para testar com uma imagem real, precisamos:

1. Baixar a imagem da URL
2. Converter para escala de cinza
3. Redimensionar para 8x8 (mesmo formato do treino)
4. Inverter as cores se necessario (o dataset tem fundo escuro e digito claro)
5. Normalizar para escala 0-16
6. Passar para `prever()`

> **Nota:** Como o modelo foi treinado em imagens muito simples de baixa resolucao, o resultado pode variar com fotos reais. Imagens com fundo limpo e digito centralizado funcionam melhor.

Para instalar as dependencias necessarias: `uv add requests Pillow`

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from io import BytesIO

# Troque esta URL por qualquer imagem de um digito na internet
# Dica: use imagens com fundo branco e numero escuro e centralizado
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/27/MnistExamples.png/320px-MnistExamples.png"

# --- Baixa a imagem ---
resposta = requests.get(url, timeout=10)
imagem_original = Image.open(BytesIO(resposta.content))

# --- Pre-processa: recorta apenas o primeiro digito da imagem ---
# Para imagens de um unico digito, pode remover o crop e ir direto para o resize
largura, altura = imagem_original.size
imagem_recortada = imagem_original.crop((0, 0, altura, altura))  # pega um quadrado inicial

# --- Converte para cinza, reduz para 8x8 e normaliza ---
imagem_cinza = imagem_recortada.convert("L")          # escala de cinza
imagem_8x8 = imagem_cinza.resize((8, 8), Image.LANCZOS)  # reduz para 8x8
imagem_inv = ImageOps.invert(imagem_8x8)              # inverte: fundo escuro, digito claro
array = np.array(imagem_inv, dtype=float) / 255 * 16  # normaliza para 0-16

# --- Visualiza o antes e depois ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
ax1.imshow(imagem_cinza, cmap="gray")
ax1.set_title("Imagem original (cinza)")
ax1.axis("off")

ax2.imshow(array, cmap="gray")
ax2.set_title("Versao 8x8 (entrada da IA)")
ax2.axis("off")

plt.tight_layout()
plt.show()

# --- Faz a previsao ---
entrada = array.reshape(1, -1)  # transforma em vetor de 64 numeros
previsao = clf_digits.prever(entrada)
print(f"A rede neural disse que o digito e: {previsao[0]}")